In [11]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_openai import OpenAIEmbeddings

import gradio as gr

In [12]:
MODEL = "gpt-5.6-luna"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [13]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [14]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [5]:
retriever.invoke("I know Rag. Which companies search for that skill?")

[Document(id='479a001c-5537-486c-b007-fa875d86693d', metadata={'source': 'knowledge-base/2026-09-14_1417/companies/cyrad-solutions.md', 'doc_type': 'companies'}, page_content='---\ncompany: "Cyrad Solutions"\nresearched_at: "2026-09-14_1417"\n---\n\n# Cyrad Solutions\n\n## Overview\nCyRAD Solutions is a privately held executive search/talent acquisition firm based in Chantilly, Virginia. Its LinkedIn profile lists about 2–10 employees and says it helps clients find engineering, scientific, technology, sales, and leadership talent, including AI/ML roles. The hiring post you shared is for a New York, NY role, but the company itself is headquartered in Virginia. ([linkedin.com](https://www.linkedin.com/company/cyrad-solutions?utm_source=openai))\n\n## Products and Services\nCyRAD Solutions provides recruiting and executive search services rather than building a product. Its stated focus includes software engineering, embedded development, cyber security, artificial intelligence, machine l

In [6]:
llm.invoke("I know Rag. Which companies search for that skill?")

AIMessage(content='If by **RAG** you mean **retrieval-augmented generation**, many companies hire for it—but the job title is usually not simply “RAG Engineer.” Search for roles such as:\n\n- **LLM Engineer / Generative AI Engineer**\n- **Applied AI Engineer**\n- **Machine Learning Engineer — NLP/LLMs**\n- **AI Search Engineer**\n- **Knowledge Engineer**\n- **Conversational AI Engineer**\n- **AI Platform Engineer**\n- **ML Infrastructure Engineer**\n\n### Companies commonly hiring for RAG-related work\n\n**AI model and platform companies**\n- OpenAI\n- Anthropic\n- Google / Google DeepMind\n- Microsoft / Azure AI\n- Amazon / AWS\n- Meta\n- NVIDIA\n- Cohere\n- Mistral AI\n- Databricks\n- Snowflake\n\n**Search, vector database, and developer-tool companies**\n- Elastic\n- Pinecone\n- Weaviate\n- Milvus / Zilliz\n- MongoDB\n- Redis\n- Vespa\n- Algolia\n- Glean\n- LangChain\n- LlamaIndex\n\n**Enterprise AI companies**\n- Writer\n- Harvey\n- Perplexity\n- Hebbia\n- Kumo AI\n- Sana\n- Kore.a

In [15]:
SYSTEM_PROMPT_TEMPLATE = """

You are a friendly and knowledgeable assistant helping the user with our knowledge base about AI engineering job postings. 
Our knowledge base includes jobs that have either AI or engineering in the title, 
and we also looked for jobs that are about building applications on top of LLMs. 

If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [16]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [17]:
answer_question("I know Rag. Which companies search for that skill?", [])

'RAG usually means **Retrieval-Augmented Generation**. In the provided data, **CyRAD Solutions** appears to recruit for AI/ML engineering roles, including an **Agentic AI Engineer** position, but the client is unnamed and the posting does not explicitly confirm that it requires RAG.\n\nCompanies that commonly search for RAG skills include:\n\n- OpenAI\n- Anthropic\n- Google\n- Microsoft\n- Amazon/AWS\n- Meta\n- Databricks\n- Snowflake\n- NVIDIA\n- Palantir\n- Scale AI\n- Cohere\n- Perplexity\n- Hugging Face\n- Enterprise AI startups and consulting firms\n\nJob titles may include **LLM Engineer, Generative AI Engineer, Applied AI Engineer, ML Engineer, AI Application Engineer, or Search/Relevance Engineer** rather than “RAG Engineer.” Useful search terms are:\n\n`RAG`, `retrieval-augmented generation`, `vector search`, `embeddings`, `semantic search`, `hybrid search`, `LangChain`, `LlamaIndex`, `Rerankers`, `knowledge graphs`, and `LLM applications`.\n\nI can identify specific companies

In [18]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
